# Train Fault Type Model

Updated notebook for the new `preprocess.py`.

Pipeline:
1. Load base processed dataframe from `preprocess.py`
2. Build fault-type dataset with `only_failures=True`
3. Encode labels
4. Train/test split
5. Fit normalization config on **train only**
6. Apply normalization to train/test
7. Apply SMOTE to **train only**
8. Rename features for XGBoost safety
9. Train multiclass model
10. Evaluate and save artifacts


In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder

from imblearn.over_sampling import SMOTE

from preprocess import (
    get_processed_dataframe,
    build_fault_type_dataset,
    build_normalization_config_from_df,
    normalize_features,
    save_json,
    COMMON_FEATURES,
)


In [2]:
ARTIFACTS_DIR = Path("training/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ARTIFACTS_DIR / "fault_type_model.pkl"
LABEL_ENCODER_PATH = ARTIFACTS_DIR / "fault_type_label_encoder.pkl"
FEATURES_PATH = ARTIFACTS_DIR / "fault_type_feature_list.json"
NORMALIZATION_CONFIG_PATH = ARTIFACTS_DIR / "fault_type_normalization_config.json"
METRICS_PATH = ARTIFACTS_DIR / "fault_type_metrics.json"
TEST_CSV_PATH = ARTIFACTS_DIR / "fault_type_test_set.csv"
TEST_PREDICTIONS_CSV_PATH = ARTIFACTS_DIR / "fault_type_test_predictions.csv"


In [3]:
SAFE_FEATURE_MAP = {
    "Air temperature [K]": "air_temperature_k",
    "temp_diff": "temp_diff",
    "Rotational speed [rpm]": "rotational_speed_rpm",
    "Torque [Nm]": "torque_nm",
    "power_kw": "power_kw",
    "Tool wear [min]": "tool_wear_min",
}


In [4]:
# Load base processed dataframe from preprocess.py
df = get_processed_dataframe()

# Build dataset only from failure rows
X, y = build_fault_type_dataset(df=df, only_failures=True)

print("Fault dataset shape:", X.shape)
print("\nFault type distribution BEFORE split:")
print(y.value_counts())

X.head()


Fault dataset shape: (330, 6)

Fault type distribution BEFORE split:
fault_type
HDF    115
PWF     91
OSF     78
TWF     46
Name: count, dtype: int64


,Air temperature [K],temp_diff,Rotational speed [rpm],Torque [Nm],power_kw,Tool wear [min]
50,298.9,10.2,2861,4.6,1.378175,143
69,298.9,10.1,1410,65.7,9.700924,191
77,298.8,10.1,1455,41.3,6.292767,208
160,298.4,9.8,1282,60.7,8.149019,216
161,298.3,9.8,1412,52.3,7.733303,218


In [5]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Classes:", list(label_encoder.classes_))


Classes: ['HDF', 'OSF', 'PWF', 'TWF']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain distribution BEFORE normalization and SMOTE:")
print(pd.Series(label_encoder.inverse_transform(y_train)).value_counts())


Train shape: (264, 6)
Test shape: (66, 6)

Train distribution BEFORE normalization and SMOTE:
HDF    92
PWF    73
OSF    62
TWF    37
Name: count, dtype: int64


In [7]:
# Fit normalization ONLY on train
norm_config = build_normalization_config_from_df(X_train)

X_train_norm = normalize_features(X_train, norm_config)
X_test_norm = normalize_features(X_test, norm_config)

print("Normalization config built from train only.")
norm_config


Normalization config built from train only.


{'Air temperature [K]': {'mode': 'minmax', 'min': 297.2, 'max': 303.485},
 'temp_diff': {'mode': 'minmax',
  'min': 7.800000000000011,
  'max': 11.300000000000011},
 'Rotational speed [rpm]': {'mode': 'ratio', 'max': 2662.4},
 'Torque [Nm]': {'mode': 'ratio', 'max': 69.085},
 'power_kw': {'mode': 'ratio', 'max': 9.466438620948633},
 'Tool wear [min]': {'mode': 'ratio', 'max': 228.0}}

In [8]:
# Save normalized test set before renaming columns
test_df = X_test_norm.copy()
test_df["true_fault_type"] = label_encoder.inverse_transform(y_test)
test_df.to_csv(TEST_CSV_PATH, index=False)

print("Saved normalized test set to:", TEST_CSV_PATH)
test_df.head()


Saved normalized test set to: training\artifacts\fault_type_test_set.csv


,Air temperature [K],temp_diff,Rotational speed [rpm],Torque [Nm],power_kw,Tool wear [min],true_fault_type
4389,0.779634,0.028571,0.508939,0.652819,0.676017,0.026316,HDF
4851,1.000000,0.171429,0.511944,0.749801,0.781030,0.394737,HDF
880,0.000000,0.771429,0.463867,1.000000,1.000000,0.390351,PWF
4121,0.779634,0.228571,0.512320,0.691901,0.721247,0.934211,HDF
69,0.270485,0.657143,0.529597,0.951002,1.000000,0.837719,PWF


In [9]:
# SMOTE ONLY on train
smote = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_norm, y_train)

print("Train shape AFTER SMOTE:", X_train_resampled.shape)
print("\nTrain distribution AFTER SMOTE:")
print(pd.Series(label_encoder.inverse_transform(y_train_resampled)).value_counts())


Train shape AFTER SMOTE: (368, 6)

Train distribution AFTER SMOTE:
OSF    92
PWF    92
HDF    92
TWF    92
Name: count, dtype: int64


In [10]:
# Rename columns to safe names for XGBoost
X_train_resampled = X_train_resampled.rename(columns=SAFE_FEATURE_MAP)
X_test_final = X_test_norm.rename(columns=SAFE_FEATURE_MAP)

print("Safe feature names:")
print(list(X_train_resampled.columns))


Safe feature names:
['air_temperature_k', 'temp_diff', 'rotational_speed_rpm', 'torque_nm', 'power_kw', 'tool_wear_min']


In [11]:
model = XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.5,
    min_child_weight=2,
    reg_alpha=0.05,
    reg_lambda=1.0,
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train_resampled, y_train_resampled)
print("Fault type model trained successfully")


Fault type model trained successfully


In [12]:
y_pred = model.predict(X_test_final)
y_prob = model.predict_proba(X_test_final)

y_test_labels = label_encoder.inverse_transform(y_test)
y_pred_labels = label_encoder.inverse_transform(y_pred)

print("Classification report:")
print(classification_report(y_test_labels, y_pred_labels, digits=4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test_labels, y_pred_labels, labels=label_encoder.classes_))


Classification report:
              precision    recall  f1-score   support

         HDF     0.9583    1.0000    0.9787        23
         OSF     0.9333    0.8750    0.9032        16
         PWF     0.9474    1.0000    0.9730        18
         TWF     0.8750    0.7778    0.8235         9

    accuracy                         0.9394        66
   macro avg     0.9285    0.9132    0.9196        66
weighted avg     0.9379    0.9394    0.9377        66


Confusion matrix:
[[23  0  0  0]
 [ 1 14  0  1]
 [ 0  0 18  0]
 [ 0  1  1  7]]


In [13]:
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

metrics = {
    "model_type": "fault_type_multiclass",
    "model_name": "XGBoost + normalization(train-only) + SMOTE(train-only)",
    "classes": list(label_encoder.classes_),
    "macro_f1": round(float(macro_f1), 6),
    "weighted_f1": round(float(weighted_f1), 6),
    "confusion_matrix": confusion_matrix(
        y_test_labels,
        y_pred_labels,
        labels=list(label_encoder.classes_)
    ).tolist(),
    "features": list(COMMON_FEATURES),
    "safe_features": list(X_train_resampled.columns),
    "train_size_before_smote": int(len(X_train)),
    "train_size_after_smote": int(len(X_train_resampled)),
    "test_size": int(len(X_test_final)),
}

print(json.dumps(metrics, ensure_ascii=False, indent=2))


{
  "model_type": "fault_type_multiclass",
  "model_name": "XGBoost + normalization(train-only) + SMOTE(train-only)",
  "classes": [
    "HDF",
    "OSF",
    "PWF",
    "TWF"
  ],
  "macro_f1": 0.919613,
  "weighted_f1": 0.93769,
  "confusion_matrix": [
    [
      23,
      0,
      0,
      0
    ],
    [
      1,
      14,
      0,
      1
    ],
    [
      0,
      0,
      18,
      0
    ],
    [
      0,
      1,
      1,
      7
    ]
  ],
  "features": [
    "Air temperature [K]",
    "temp_diff",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "power_kw",
    "Tool wear [min]"
  ],
  "safe_features": [
    "air_temperature_k",
    "temp_diff",
    "rotational_speed_rpm",
    "torque_nm",
    "power_kw",
    "tool_wear_min"
  ],
  "train_size_before_smote": 264,
  "train_size_after_smote": 368,
  "test_size": 66
}


In [14]:
proba_df = pd.DataFrame(
    y_prob,
    columns=[f"prob_{cls}" for cls in label_encoder.classes_]
)

test_predictions_df = X_test_norm.reset_index(drop=True).copy()
test_predictions_df["true_fault_type"] = y_test_labels
test_predictions_df["predicted_fault_type"] = y_pred_labels
test_predictions_df = pd.concat([test_predictions_df, proba_df], axis=1)
test_predictions_df.to_csv(TEST_PREDICTIONS_CSV_PATH, index=False)

print("Saved test predictions to:", TEST_PREDICTIONS_CSV_PATH)
test_predictions_df.head()


Saved test predictions to: training\artifacts\fault_type_test_predictions.csv


,Air temperature [K],temp_diff,Rotational speed [rpm],Torque [Nm],power_kw,Tool wear [min],true_fault_type,predicted_fault_type,prob_HDF,prob_OSF,prob_PWF,prob_TWF
0,0.779634,0.028571,0.508939,0.652819,0.676017,0.026316,HDF,HDF,0.990602,0.003029,0.002825,0.003543
1,1.000000,0.171429,0.511944,0.749801,0.781030,0.394737,HDF,HDF,0.990974,0.004316,0.002826,0.001884
2,0.000000,0.771429,0.463867,1.000000,1.000000,0.390351,PWF,PWF,0.003535,0.005555,0.988055,0.002856
3,0.779634,0.228571,0.512320,0.691901,0.721247,0.934211,HDF,HDF,0.914197,0.015121,0.011835,0.058847
4,0.270485,0.657143,0.529597,0.951002,1.000000,0.837719,PWF,PWF,0.002285,0.045854,0.946470,0.005391


In [15]:
feature_importance = pd.DataFrame({
    "feature": list(X_train_resampled.columns),
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance


,feature,importance
1,temp_diff,0.253274
3,torque_nm,0.246982
4,power_kw,0.182955
5,tool_wear_min,0.144799
2,rotational_speed_rpm,0.093908
0,air_temperature_k,0.078082


In [16]:
joblib.dump(model, MODEL_PATH)
joblib.dump(label_encoder, LABEL_ENCODER_PATH)
save_json(list(COMMON_FEATURES), FEATURES_PATH)
save_json(norm_config, NORMALIZATION_CONFIG_PATH)

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Saved model to:", MODEL_PATH)
print("Saved label encoder to:", LABEL_ENCODER_PATH)
print("Saved feature list to:", FEATURES_PATH)
print("Saved normalization config to:", NORMALIZATION_CONFIG_PATH)
print("Saved metrics to:", METRICS_PATH)


Saved model to: training\artifacts\fault_type_model.pkl
Saved label encoder to: training\artifacts\fault_type_label_encoder.pkl
Saved feature list to: training\artifacts\fault_type_feature_list.json
Saved normalization config to: training\artifacts\fault_type_normalization_config.json
Saved metrics to: training\artifacts\fault_type_metrics.json
